# Lab 04 — Fault Tolerance: Drive-Level Resilience

This lab demonstrates how RustFS tolerates drive failures using **Erasure Coding (EC:2)**.

Our `docker-compose.yml` mounts **4 Docker volumes** (`drive0`…`drive3`) as separate drives
inside a single RustFS container. RustFS applies a **Reed-Solomon (4+2)** scheme:
- Every object is split into **4 data shards** + **2 parity shards**
- The 6 shards are distributed across all 4 drives
- RustFS can **lose any 2 drives** and still reconstruct every object

In this lab you'll upload objects, verify they're intact, inspect the container health,
and confirm the theoretical protection guarantees.


---
## 0 · Prerequisites

1. RustFS is running: `docker compose up -d`
2. Virtual environment active with `boto3` installed: `uv sync`
3. Docker CLI available in PATH (for health checks)


---
## Setup — Connect and Create Bucket


In [ ]:
import subprocess

import boto3
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
    ),
)

BUCKET = 'lab4-ft'

existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f'✅ Created bucket: {BUCKET}')
else:
    print(f'✅ Bucket already exists: {BUCKET}')


---
## 1 · Upload Test Objects

We upload 10 small text objects. RustFS will automatically stripe each object
across the 4 drives as 4 data shards + 2 parity shards.


In [ ]:
NUM_OBJECTS = 10

for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    content = f'Object {i:02d}: RustFS fault tolerance test. Shard distributed across 4 drives.'
    s3.put_object(Bucket=BUCKET, Key=key, Body=content.encode('utf-8'))
    print(f'  Uploaded {key}')

print(f'\n✅ {NUM_OBJECTS} objects uploaded to s3://{BUCKET}/')


---
## 2 · Verify All Objects Are Intact

Read back all objects and verify the content matches exactly.
This confirms that write → EC stripe → read → EC reconstruct works correctly.


In [ ]:
print('Verifying objects...')
all_ok = True
for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    response = s3.get_object(Bucket=BUCKET, Key=key)
    body = response['Body'].read().decode('utf-8')
    expected = f'Object {i:02d}: RustFS fault tolerance test. Shard distributed across 4 drives.'
    ok = body == expected
    status = '✅' if ok else '❌'
    print(f'  {status} {key}')
    if not ok:
        all_ok = False
        print(f'     Expected: {expected}')
        print(f'     Got:      {body}')

print(f'\n{"✅ All objects verified intact." if all_ok else "❌ Verification failed!"}')


---
## 3 · Check Server Health

RustFS exposes a liveness endpoint at `/minio/health/live` (HTTP 200 = healthy).
We also run `docker compose ps` to confirm all containers are running.


In [ ]:
# --- Liveness check via HTTP ---
result = subprocess.run(
    ['curl', '-sf', '-o', '/dev/null', '-w', '%{http_code}',
     'http://localhost:9000/minio/health/live'],
    capture_output=True, text=True,
)
status_code = result.stdout.strip()
if status_code == '200':
    print('✅ RustFS liveness check: HTTP 200 — server is healthy')
else:
    print(f'⚠️  Unexpected HTTP status: {status_code}')

print()

# --- Container status via Docker ---
result = subprocess.run(
    ['docker', 'compose', 'ps'],
    capture_output=True, text=True, cwd='..'
)
print('Container status:')
print(result.stdout if result.stdout else '(no output — run from repo root)')


---
## Understanding the EC:2 Protection Model

RustFS uses **Reed-Solomon (4+2)** erasure coding inside a single server with 4 drives:

```
Object bytes  ──▶  [D1][D2][D3][D4]  ← 4 data shards
                   [P1][P2]           ← 2 parity shards (XOR + RS math)
                    │   │   │   │
                drive0 drive1 drive2 drive3  (Docker volumes)
```

The math guarantees: **any 4 of the 6 shards are sufficient to reconstruct the object.**
That means losing 2 drives still leaves 4 surviving shards → full reconstruction.

| Drives lost | Shards remaining | Data recoverable? |
|-------------|-----------------|------------------|
| 0 | 6 / 6 | Yes (normal read) |
| 1 | 5 / 6 | Yes (EC reconstructs) |
| 2 | 4 / 6 | Yes (EC:2 tolerance limit) |
| 3 | 3 / 6 | No — insufficient shards |

Storage efficiency: **4 data shards / 6 total = 67%** (vs 33% for 3× replication).

In this single-node lab, the 4 drives are Docker volumes on one machine.
In a production cluster, shards are distributed across **different physical nodes**,
so drive AND node failures are both tolerated.


---
## 4 · Resilience in Action — Simulating Drive Failures

We simulate failures by making a drive's shards inaccessible via **renaming
the bucket directory on that drive**. RustFS detects `ENOENT` on the first I/O
and transparently reconstructs the object from surviving shards + parity.

The demo walks through three stages to exercise the full RS(4,2) tolerance boundary:

| Stage | Drives offline | Shards remaining | Recoverable? |
|-------|---------------|------------------|--------------|
| 1     | 1 of 4        | 5 / 6            | ✅ Yes |
| 2     | 2 of 4        | 4 / 6            | ✅ Yes (RS 4+2 limit) |
| 3     | 3 of 4        | 3 / 6            | ❌ No — insufficient shards |

In [ ]:
print('Shard layout — files per drive:\n')
for n in range(4):
    r = subprocess.run(
        ['docker', 'exec', 'rustfs-server', 'find', f'/data/drive{n}/{BUCKET}', '-type', 'f'],
        capture_output=True, text=True,
    )
    count = len([line for line in r.stdout.strip().splitlines() if line])
    status = 'OK' if r.returncode == 0 else 'EMPTY/MISSING'
    print(f'  drive{n}: {count} files  [{status}]')

---
### Stage 1 — 1 Drive Failure (within RS 4+2 tolerance)

5 of 6 shards survive. RustFS reconstructs the missing shard from parity.
Failure is detected on the first `GET` — no polling, no server restart.

In [ ]:
r = subprocess.run(
    ['docker', 'exec', 'rustfs-server',
     'mv', f'/data/drive1/{BUCKET}', f'/data/drive1/{BUCKET}_FAILED'],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print('SIMULATED: drive1 data inaccessible (bucket directory renamed)')
    print('RustFS was NOT restarted — failure will be detected on next I/O')
else:
    print(f'Error: {r.stderr}')

In [ ]:
print('Reading all objects with drive1 offline (1/4 drives failed):\n')
errors = 0
for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    try:
        body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read().decode('utf-8')
        expected = f'Object {i:02d}: RustFS fault tolerance test. Shard distributed across 4 drives.'
        icon = '✅' if body == expected else '❌ WRONG CONTENT'
        print(f'  [{icon}] {key}: reconstructed from surviving shards')
    except Exception as e:
        print(f'  [❌ FAIL] {key}: {e}')
        errors += 1

print(f'\n{"✅ All 10 objects accessible — EC reconstructed from 3 surviving drives!" if errors == 0 else f"❌ {errors} failures"}')

---
### Stage 2 — 2 Drive Failures (at the RS 4+2 limit)

4 of 6 shards survive — the minimum required (`k=4`). Still recoverable, but
any additional failure would cause data loss.

In [ ]:
r = subprocess.run(
    ['docker', 'exec', 'rustfs-server',
     'mv', f'/data/drive2/{BUCKET}', f'/data/drive2/{BUCKET}_FAILED'],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print('SIMULATED: drive2 now inaccessible (2 of 4 drives offline)')
else:
    print(f'Error: {r.stderr}')

In [ ]:
print('Reading all objects with drive1 + drive2 offline (2/4 drives failed):\n')
errors = 0
for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    try:
        body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read().decode('utf-8')
        expected = f'Object {i:02d}: RustFS fault tolerance test. Shard distributed across 4 drives.'
        icon = '✅' if body == expected else '❌ WRONG CONTENT'
        print(f'  [{icon}] {key}')
    except Exception as e:
        print(f'  [❌ FAIL] {key}: {e}')
        errors += 1

print(f'\nResult: {"✅ Accessible — RS(4,2) limit reached but not exceeded!" if errors == 0 else f"❌ {errors}/10 objects lost"}')

---
### Stage 3 — 3 Drive Failures (beyond tolerance — expected data loss)

RS(4,2) requires at least 4 shards to reconstruct. Losing 3 drives leaves only
3 surviving shards — insufficient. **This is where the EC guarantee breaks down.**

In [ ]:
r = subprocess.run(
    ['docker', 'exec', 'rustfs-server',
     'mv', f'/data/drive3/{BUCKET}', f'/data/drive3/{BUCKET}_FAILED'],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print('SIMULATED: drive3 now inaccessible (3 of 4 drives offline)')
    print('WARNING: this exceeds RS(4,2) tolerance — expect read failures')
else:
    print(f'Error: {r.stderr}')

In [ ]:
print('Reading all objects with drive1+drive2+drive3 offline (3/4 drives failed):\n')
accessible, failed = 0, 0
for i in range(1, NUM_OBJECTS + 1):
    key = f'file_{i:02d}.txt'
    try:
        s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
        print(f'  [✅] {key}')
        accessible += 1
    except Exception as e:
        print(f'  [❌] {key}: {type(e).__name__}')
        failed += 1

print(f'\n{accessible} accessible, {failed} failed — beyond RS(4,2) tolerance')

In [ ]:
print('Restoring all drives (renaming back)...\n')
for n in [1, 2, 3]:
    r = subprocess.run(
        ['docker', 'exec', 'rustfs-server', 'sh', '-c',
         f'test -d /data/drive{n}/{BUCKET}_FAILED && '
         f'mv /data/drive{n}/{BUCKET}_FAILED /data/drive{n}/{BUCKET} && '
         f'echo "drive{n}: restored" || echo "drive{n}: nothing to restore"'],
        capture_output=True, text=True,
    )
    print(f'  {r.stdout.strip()}')

print('\nVerifying all objects after restore:')
errors = 0
for i in range(1, NUM_OBJECTS + 1):
    try:
        s3.get_object(Bucket=BUCKET, Key=f'file_{i:02d}.txt')['Body'].read()
    except Exception:
        errors += 1
print(f'  {"✅ All 10 objects verified — drives restored successfully!" if errors == 0 else f"❌ {errors} failures after restore"}')

---
## 5 · Cleanup


In [ ]:
# Delete all objects
response = s3.list_objects_v2(Bucket=BUCKET)
for obj in response.get('Contents', []):
    s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
    print(f'🗑️  Deleted {obj["Key"]}')

# Delete the bucket
s3.delete_bucket(Bucket=BUCKET)
print(f'🗑️  Deleted bucket: {BUCKET}')

print('\n✅ Cleanup complete!')


---
## 📋 Summary

| Concept | Value in this lab |
|---------|-------------------|
| EC scheme | Reed-Solomon (4+2) |
| Data shards | 4 |
| Parity shards | 2 |
| Max drives tolerated | 2 |
| Storage efficiency | 4/6 = 67% |
| vs 3× replication | 67% vs 33% (2× more efficient) |

### How to simulate a drive failure

Rename the bucket directory on the target drive — immediate detection on next GET:
```bash
# Simulate drive1 failure
docker exec rustfs-server mv /data/drive1/BUCKET /data/drive1/BUCKET_FAILED
# ... reads still work via EC ...
# Restore
docker exec rustfs-server mv /data/drive1/BUCKET_FAILED /data/drive1/BUCKET
```

> **Why not `chmod 000`?** The RustFS process may keep the directory open via a
> cached file descriptor, making the permission change ineffective. A rename forces
> `ENOENT` on the next path lookup, guaranteeing immediate detection.

### Next steps

- **Lab 05**: Object versioning — protect data from accidental overwrites
- **Lab 06**: Erasure Coding deep dive — efficiency vs fault tolerance trade-offs